# Decoding

In [1]:
# !pip uninstall -y protobuf
# !pip install protobuf==3.20.3

In [1]:
import pandas as pd
events = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-events-1-2025-to-3-2025.csv')

KeyboardInterrupt: 

In [ ]:
# Import your protobuf classes
from ctl.iox import (
    location_pb2,
    readings_pb2,
    alarm_pb2,
    pf_instrument_config_pb2,
    cal_record_pb2,
    inst_mode_pb2,
    inst_debug_pb2,
    inst_cloud_response_pb2,
    cico_pb2,
    battery_pb2,
    sensor_states_record_pb2,
    tag_pb2,
    connectivity_pb2
)

ModuleNotFoundError: No module named 'ctl'

In [ ]:
EVENT_TYPE_TO_PROTO = {
    # Alarm-related
    "ALARM": alarm_pb2.AlarmRecord,
    "WARNING": alarm_pb2.AlarmRecord,
    "NOTIFICATION": alarm_pb2.AlarmRecord,

    # Battery
    "BATTERY": battery_pb2.BatteryRecord,
    "BATTERY_INFO": tag_pb2.BatteryPayload,

    # Location
    "LOCATION": location_pb2.LocationRecord,

    # Calibration / config
    "CALIBRATION": cal_record_pb2.CalRecord,

    # Cloud / connectivity
    "CLOUD_RESPONSE": inst_cloud_response_pb2.CloudResponse,
    "CONNECTIVITY": connectivity_pb2.ConnectivityMsg,  # confirm exists

    # Tag / CICO
    "CICO": cico_pb2.TagRecord,

    # Instrument state
    "MODE": inst_mode_pb2.InstrumentModeRecord,

    # Sensor readings (if present elsewhere)
    "READINGS": readings_pb2.ReadingsRecord,

    # Legacy / unsupported
    "GENERIC": None,        # legacy 5-star link
    "GRID_ACTION": None,    # no pb2 provided
}


In [ ]:
import base64
import math
import pandas as pd

def decode_by_event_type(encoded_proto, event_type):
    if pd.isna(encoded_proto) or not event_type:
        return None

    proto_cls = EVENT_TYPE_TO_PROTO.get(event_type)

    # Explicitly skip unsupported / legacy events
    if proto_cls is None:
        return None

    try:
        # Decode base64
        if isinstance(encoded_proto, (bytes, bytearray)):
            raw = encoded_proto
        else:
            raw = base64.b64decode(encoded_proto)

        # Parse protobuf
        msg = proto_cls()
        msg.ParseFromString(raw)
        return msg

    except Exception as e:
        # Fail loudly but safely
        print(f"Failed to decode event_type={event_type}: {e}")
        return None

events["decoded_message"] = events.apply(
    lambda row: decode_by_event_type(
        row["META_ENCODED_PROTO"],
        row["EVENT_TYPE"]
    ),
    axis=1
)

events["decoded_type"] = events["decoded_message"].apply(
    lambda x: x.DESCRIPTOR.full_name if x else None
)
